# 01 · Exploratory data analysis

CRISP-DM phase 2 — *data understanding*.

This notebook works through the source data to answer three questions:

1. What does the target distribution look like, and what does that imply for the loss?
2. Which signals actually carry information about trip duration?
3. What is dirty, and how much of it is there?

Findings here become the cleaning rules and features in
[`03-data-preparation.md`](../docs/03-data-preparation.md).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nyctaxi.clean import clean, haversine_km
from nyctaxi.config import get_config
from nyctaxi.data.loader import load

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18, "font.size": 9,
})
ACCENT, GOOD = "#e5a000", "#128a5b"
cfg = get_config()
print("config loaded:", cfg.project_name)

## Load

The loader picks Kaggle if credentials are present and TLC otherwise, then
normalises either into the same canonical schema. We take a small sample —
this notebook is for looking, not training.

In [ ]:
result = load(sample_frac=0.01)
raw = result.df
print(f"source: {result.source}")
for note in result.notes:
    print(f"  - {note}")
print()
print(f"{len(raw):,} rows, {raw['pickup_datetime'].min()} .. {raw['pickup_datetime'].max()}")
raw.head()

In [ ]:
raw.describe(include="all").T

## The target is strongly right-skewed

Most trips are short; a long tail runs for an hour or more. Two consequences
follow directly, and both are load-bearing decisions elsewhere in the project:
we train on `log1p(duration)`, and we score with RMSLE rather than RMSE.

In [ ]:
dur = raw["trip_duration_s"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

axes[0].hist(dur[dur < 5000] / 60, bins=80, color=ACCENT)
axes[0].set(title="Trip duration (raw)", xlabel="minutes", ylabel="trips")

axes[1].hist(np.log1p(dur), bins=80, color=GOOD)
axes[1].set(title="log1p(duration) — near-symmetric", xlabel="log1p(seconds)")

plt.tight_layout()
print(dur.describe(percentiles=[.01, .25, .5, .75, .99]).round(1))

## Data quality: where the dirt is

Four failure modes recur. Each becomes a rule in `nyctaxi.clean`, and each rule
reports how many rows it removed so the cleaning story is auditable rather than
asserted.

In [ ]:
km = haversine_km(
    raw["pickup_lat"].to_numpy(), raw["pickup_lon"].to_numpy(),
    raw["dropoff_lat"].to_numpy(), raw["dropoff_lon"].to_numpy(),
)
speed = km / (raw["trip_duration_s"].to_numpy() / 3600)

checks = {
    "under 30 seconds":     (raw["trip_duration_s"] < 30).sum(),
    "over 3 hours":         (raw["trip_duration_s"] > 10800).sum(),
    "implied speed > 100kmh": (speed > 100).sum(),
    "implied speed < 1kmh":   (speed < 1).sum(),
    "zero straight-line distance": (km < 0.01).sum(),
}
for name, n in checks.items():
    print(f"  {name:32s} {n:6,}  ({100 * n / len(raw):.3f}%)")

In [ ]:
clean_df, report = clean(raw.copy())
print(report.to_markdown())

## Hour of day is the strongest non-geometric signal

The same route takes wildly different times depending on when you leave. This
is why `hour`, its cyclic encoding, and the hour-keyed speed aggregates all
exist as features — and why the app has a 24-hour departure curve at all.

In [ ]:
d = clean_df.assign(
    hour=clean_df["pickup_datetime"].dt.hour,
    weekday=clean_df["pickup_datetime"].dt.dayofweek,
    minutes=clean_df["trip_duration_s"] / 60,
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
by_hour = d.groupby("hour")["minutes"].median()
axes[0].plot(by_hour.index, by_hour.to_numpy(), color=ACCENT, lw=2, marker="o", ms=3)
axes[0].set(title="Median duration by hour", xlabel="hour", ylabel="minutes")

pivot = d.pivot_table(index="weekday", columns="hour", values="minutes", aggfunc="median")
im = axes[1].imshow(pivot, aspect="auto", cmap="magma", origin="lower")
axes[1].set(title="Median duration: weekday x hour", xlabel="hour", ylabel="weekday (0=Mon)")
plt.colorbar(im, ax=axes[1], label="minutes")
plt.tight_layout()

peak, quiet = by_hour.idxmax(), by_hour.idxmin()
print(f"slowest {peak}:00 ({by_hour[peak]:.1f} min) vs quickest {quiet}:00 ({by_hour[quiet]:.1f} min)"
      f"  ->  {by_hour[peak] / by_hour[quiet]:.2f}x")

## Distance explains a lot — but nowhere near everything

The spread at any fixed distance is what the rest of the feature set is for.
If distance were sufficient, the physics baseline would not lose to LightGBM
by such a wide margin.

In [ ]:
d = d.assign(km=haversine_km(
    d["pickup_lat"].to_numpy(), d["pickup_lon"].to_numpy(),
    d["dropoff_lat"].to_numpy(), d["dropoff_lon"].to_numpy(),
))
sub = d[d["km"] < 25]

fig, ax = plt.subplots(figsize=(6, 4))
ax.hexbin(sub["km"], sub["minutes"].clip(upper=90), gridsize=45, cmap="magma", mincnt=1)
ax.set(title="Duration vs straight-line distance", xlabel="km", ylabel="minutes")
plt.tight_layout()

print("correlation, distance vs duration:", round(sub["km"].corr(sub["minutes"]), 3))
print("correlation, in log space:       ",
      round(np.log1p(sub["km"]).corr(np.log1p(sub["minutes"])), 3))

## Where trips start

On the TLC path these are points sampled inside taxi-zone polygons, so this
plot shows zone *shapes* rather than true pickup density. On the Kaggle path it
is genuine GPS, and Manhattan's street grid is visible.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 6.8))
ax.scatter(clean_df["pickup_lon"], clean_df["pickup_lat"], s=0.4, alpha=0.12,
           color=ACCENT, linewidths=0)
ax.set(title=f"Pickup locations ({result.source} source)", xlabel="longitude", ylabel="latitude",
       xlim=(-74.05, -73.75), ylim=(40.58, 40.9))
ax.set_aspect(1.32)
ax.grid(alpha=0.1)
plt.tight_layout()

## What this establishes

- **Right-skewed target** → train on `log1p`, score with RMSLE.
- **Hour of day matters, and more than it looks here.** The *aggregate* median
  swings about 1.4x across the day, but that mixes short and long trips
  together. Hold the route fixed and the effect is far larger — the trained
  model puts Times Square to JFK at ~31 min at 5am and ~70 min at 5:30pm, a
  2.3x swing (see `02_modeling.ipynb`). Time features and the hour-keyed speed
  aggregates are essential.
- **Distance is necessary, not sufficient** → the wide spread at fixed distance
  is the headroom the model exploits.
- **Under 1% of rows are dirty**, concentrated in exactly the tails a squared
  loss is most sensitive to → cleaning is cheap and worth doing.

→ Continue in [`02_modeling.ipynb`](02_modeling.ipynb).